### Test Random Env with OpenAI Gymnasium

In [1]:
import gymnasium as gym
import random

In [25]:
visual_env = gym.make("CartPole-v1", render_mode='human')
training_env = gym.make("CartPole-v1")
states = env.observation_space.shape[0]
actions = int(env.action_space.n)

In [12]:
actions

2

In [19]:
episodes = 10
for episode in range(1, episodes + 1):
    state = visual_env.reset()
    done = False
    score = 0

    while not done:
        visual_env.render()
        action = random.choice([0,1])
        n_state, reward, terminated, truncated, info = visual_env.step(action)
        score += reward
        done = terminated or truncated
    print(f"Episode {episode}: Score: {score}")

Episode 1: Score: 36.0
Episode 2: Score: 30.0
Episode 3: Score: 37.0
Episode 4: Score: 14.0
Episode 5: Score: 20.0
Episode 6: Score: 17.0
Episode 7: Score: 12.0
Episode 8: Score: 13.0
Episode 9: Score: 22.0
Episode 10: Score: 72.0


### Create Deep Learning Model with Keras

In [20]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.optimizers import Adam

In [21]:
def build_model(states, actions):
    model = Sequential()
    model.add(Input(shape=(1, states)))
    model.add(Flatten())
    model.add(Dense(24, activation='relu'))
    model.add(Dense(24, activation='relu'))
    model.add(Dense(actions, activation='linear'))
    return model

In [22]:
model = build_model(states, actions)
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 4)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │           120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 24)             │           600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 2)              │            50 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 770 (3.01 KB)

 Trainable params: 770 (3.01 KB)

 Non-trainable params: 0 (0.00 B)

### Build Agent with Keras-RL

#### keras-rl2 compatibility note

`keras-rl2==1.0.5` predates the current Python/TensorFlow/Keras/Gymnasium stack used by this notebook. I patched this package in my virtual environment locally, but for you these patches will not be present.

If you install `keras-rl2` into a new virtual environment and run into the errors below, either patch the installed `rl` package in that environment or use an older dependency set where `keras-rl2` is known to work.

Patched library issues (you need to patch these yourself):

- Import failure: Keras 3 removed `model_from_config`, so `rl.util.clone_model` now uses Keras 3's supported model cloning API.
- Import side effect: `rl.agents.ddpg` disabled eager execution globally during import, which broke Keras 3 weight cloning; that import-time side effect was removed.
- Callback mismatch: `rl.callbacks` used private `tensorflow.python.keras` callback classes; it now uses public Keras callback and progress bar imports.
- DQN model access: old DQN code assumed `model.input`, `model.output`, and unconditional `reset_states()` behavior; it was patched for Keras 3 Sequential/Functional models and non-stateful models.
- Gymnasium API: `reset()` and `step()` return newer tuple shapes; `rl.core` now adapts Gymnasium results to the old Gym shape expected by `keras-rl2`.
- DQN state batches: state batches are normalized to numeric arrays so TensorFlow can convert them during prediction/training.
- Soft target updates: Keras 3 no longer supports the old optimizer `get_updates` path used by `keras-rl2`; DQN soft target updates are applied after train batches instead.
- Training logs: Keras 3 returns `loss`, `mae`, and `mean_q` values during updates; DQN metric names were patched so interval logging does not crash with an inhomogeneous metrics array.
- Long runs: `Agent.fit()` used `np.int16` counters, which overflow before 50,000 steps; training counters now use normal Python integers.
- Visualization: the visualizer called `env.render(mode='human')`, but Gymnasium selects render mode when the environment is created; the visualizer now falls back to `env.render()`.

After applying or reapplying any library patch, restart the notebook kernel before running the cells below.

In [23]:
from rl.agents import DQNAgent
from rl.policy import BoltzmannQPolicy
from rl.memory import SequentialMemory

In [24]:
def build_agent(model, actions):
    policy = BoltzmannQPolicy()
    memory = SequentialMemory(limit=50000, window_length=1)
    dqn = DQNAgent(
        model=model,
        memory=memory,
        policy=policy,
        nb_actions=actions,
        nb_steps_warmup=1000,
        target_model_update=1e-2
    )
    return dqn

In [28]:
dqn = build_agent(model, actions)
dqn.compile(Adam(learning_rate=1e-3), metrics=['mae'])
dqn.fit(training_env, nb_steps=50000, visualize=False, verbose=1)

Training for 50000 steps ...
Interval 1 (0 steps performed)
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 11s 1ms/step - reward: 1.0000
26 episodes - episode_reward: 381.077 [185.000, 500.000] - loss: 1.877 - mae: 44.676 - mean_q: 90.411

Interval 2 (10000 steps performed)
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 12s 1ms/step - reward: 1.0000
28 episodes - episode_reward: 360.071 [48.000, 500.000] - loss: 8.330 - mae: 52.869 - mean_q: 106.378

Interval 3 (20000 steps performed)
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 13s 1ms/step - reward: 1.0000
60 episodes - episode_reward: 164.917 [12.000, 449.000] - loss: 16.383 - mae: 57.254 - mean_q: 114.940

Interval 4 (30000 steps performed)
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 14s 1ms/step - reward: 1.0000
48 episodes - episode_reward: 209.875 [54.000, 500.000] - loss: 18.192 - mae: 58.856 - mean_q: 118.082

Interval 5 (40000 steps performed)
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 14s 1ms/step - reward: 1.0000
done, took 64.216 seconds


In [30]:
scores = dqn.test(training_env, nb_episodes=100, visualize=False)
print(np.mean(scores.history['episode_reward']))

Testing for 100 episodes ...
Episode 1: reward: 500.000, steps: 500
Episode 2: reward: 500.000, steps: 500
Episode 3: reward: 500.000, steps: 500
Episode 4: reward: 500.000, steps: 500
Episode 5: reward: 500.000, steps: 500
Episode 6: reward: 500.000, steps: 500
Episode 7: reward: 500.000, steps: 500
Episode 8: reward: 500.000, steps: 500
Episode 9: reward: 500.000, steps: 500
Episode 10: reward: 500.000, steps: 500
Episode 11: reward: 500.000, steps: 500
Episode 12: reward: 500.000, steps: 500
Episode 13: reward: 500.000, steps: 500
Episode 14: reward: 500.000, steps: 500
Episode 15: reward: 500.000, steps: 500
Episode 16: reward: 500.000, steps: 500
Episode 17: reward: 500.000, steps: 500
Episode 18: reward: 500.000, steps: 500
Episode 19: reward: 500.000, steps: 500
Episode 20: reward: 500.000, steps: 500
Episode 21: reward: 500.000, steps: 500
Episode 22: reward: 500.000, steps: 500
Episode 23: reward: 500.000, steps: 500
Episode 24: reward: 500.000, steps: 500
Episode 25: reward: 

In [31]:
_ = dqn.test(visual_env, nb_episodes=10, visualize=True)

Testing for 10 episodes ...
Episode 1: reward: 500.000, steps: 500
Episode 2: reward: 500.000, steps: 500
Episode 3: reward: 500.000, steps: 500
Episode 4: reward: 500.000, steps: 500
Episode 5: reward: 500.000, steps: 500
Episode 6: reward: 500.000, steps: 500
Episode 7: reward: 500.000, steps: 500
Episode 8: reward: 500.000, steps: 500
Episode 9: reward: 500.000, steps: 500
Episode 10: reward: 500.000, steps: 500


### Reloading Agent from Memory

In [33]:
dqn.save_weights('dqn_cart_pole_v1.weights.h5', overwrite=True)

In [36]:
del model
del dqn
del visual_env
del training_env
# Says error because I already deleted them

NameError: name 'model' is not defined

In [37]:
del training_env

In [38]:
dqn.test(visual_env, nb_episodes=5, visualize=True)

NameError: name 'dqn' is not defined

In [40]:
visual_env = gym.make('CartPole-v1', render_mode='human')
actions = int(visual_env.action_space.n)
states = visual_env.observation_space.shape[0]
model = build_model(states, actions)
dqn = build_agent(model, actions)
dqn.compile(Adam(learning_rate=1e-3), metrics=['mae'])

In [41]:
dqn.load_weights("dqn_cart_pole_v1.weights.h5")

In [42]:
_ = dqn.test(visual_env, nb_episodes=5, visualize=True)

Testing for 5 episodes ...
Episode 1: reward: 500.000, steps: 500
Episode 2: reward: 500.000, steps: 500
Episode 3: reward: 500.000, steps: 500
Episode 4: reward: 500.000, steps: 500
Episode 5: reward: 500.000, steps: 500
